In [7]:
import pandas as pd

In [8]:
df = pd.read_csv("../data/paysim.csv")

In [9]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [11]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [12]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [13]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [14]:
fraud_df = df[df["isFraud"] == 1]

fraud_df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0
251,1,TRANSFER,2806.0,C1420196421,2806.0,0.0,C972765878,0.0,0.0,1,0
252,1,CASH_OUT,2806.0,C2101527076,2806.0,0.0,C1007251739,26202.0,0.0,1,0
680,1,TRANSFER,20128.0,C137533655,20128.0,0.0,C1848415041,0.0,0.0,1,0


In [15]:
fraud_df["type"].value_counts()

type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

In [16]:
fraud_df[["amount"]].describe()

,amount
count,8.213000e+03
mean,1.467967e+06
std,2.404253e+06
min,0.000000e+00
25%,1.270913e+05
50%,4.414234e+05
75%,1.517771e+06
max,1.000000e+07


In [17]:
high_risk_types = ["TRANSFER", "CASH_OUT"]

df["risk_type"] = df["type"].apply(
    lambda x: 1 if x in high_risk_types else 0
)

df[["type", "risk_type"]].head()

,type,risk_type
0,PAYMENT,0
1,PAYMENT,0
2,TRANSFER,1
3,CASH_OUT,1
4,PAYMENT,0


In [18]:
df["high_amount"] = df["amount"].apply(
    lambda x: 1 if x > 1000000 else 0
)

df[["amount", "high_amount"]].head()

,amount,high_amount
0,9839.64,0
1,1864.28,0
2,181.00,0
3,181.00,0
4,11668.14,0


In [22]:
df["risk_score"] = (
    df["risk_type"] * 30 +
    df["high_amount"] * 25 +
    df["balance_drained"] * 35
)

df["decision"] = df["risk_score"].apply(decision)

df[["type", "amount", "oldbalanceOrg", "newbalanceOrig", "risk_score", "decision", "isFraud"]].head(20)

,type,amount,oldbalanceOrg,newbalanceOrig,risk_score,decision,isFraud
0,PAYMENT,9839.64,170136.00,160296.36,0,ALLOW,0
1,PAYMENT,1864.28,21249.00,19384.72,0,ALLOW,0
2,TRANSFER,181.00,181.00,0.00,65,BLOCK,1
3,CASH_OUT,181.00,181.00,0.00,65,BLOCK,1
4,PAYMENT,11668.14,41554.00,29885.86,0,ALLOW,0
5,PAYMENT,7817.71,53860.00,46042.29,0,ALLOW,0
6,PAYMENT,7107.77,183195.00,176087.23,0,ALLOW,0
7,PAYMENT,7861.64,176087.23,168225.59,0,ALLOW,0
8,PAYMENT,4024.36,2671.00,0.00,35,REVIEW,0
9,DEBIT,5337.77,41720.00,36382.23,0,ALLOW,0


In [25]:
def decision(score):
    if score >= 80:
        return "BLOCK"
    elif score >= 50:
        return "REVIEW"
    else:
        return "ALLOW"

df["decision"] = df["risk_score"].apply(decision)

df[["type", "amount", "risk_score", "decision"]].head(10)

,type,amount,risk_score,decision
0,PAYMENT,9839.64,0,ALLOW
1,PAYMENT,1864.28,0,ALLOW
2,TRANSFER,181.00,65,REVIEW
3,CASH_OUT,181.00,65,REVIEW
4,PAYMENT,11668.14,0,ALLOW
5,PAYMENT,7817.71,0,ALLOW
6,PAYMENT,7107.77,0,ALLOW
7,PAYMENT,7861.64,0,ALLOW
8,PAYMENT,4024.36,35,ALLOW
9,DEBIT,5337.77,0,ALLOW


In [21]:
df["balance_drained"] = (
    (df["oldbalanceOrg"] > 0) &
    (df["newbalanceOrig"] == 0)
).astype(int)

df[["oldbalanceOrg", "newbalanceOrig", "balance_drained"]].head(10)

,oldbalanceOrg,newbalanceOrig,balance_drained
0,170136.00,160296.36,0
1,21249.00,19384.72,0
2,181.00,0.00,1
3,181.00,0.00,1
4,41554.00,29885.86,0
5,53860.00,46042.29,0
6,183195.00,176087.23,0
7,176087.23,168225.59,0
8,2671.00,0.00,1
9,41720.00,36382.23,0


In [26]:
pd.crosstab(df["isFraud"], df["decision"])


decision,ALLOW,BLOCK,REVIEW
isFraud,,,
0,5109462,62918,1182027
1,43,2548,5622


In [24]:
df[df["decision"] == "BLOCK"][[
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "risk_score",
    "isFraud"
]].head(20)

,type,amount,oldbalanceOrg,newbalanceOrig,risk_score,isFraud
2,TRANSFER,181.00,181.00,0.0,65,1
3,CASH_OUT,181.00,181.00,0.0,65,1
15,CASH_OUT,229133.94,15325.00,0.0,65,0
19,TRANSFER,215310.30,705.00,0.0,65,0
24,TRANSFER,311685.89,10835.00,0.0,65,0
42,CASH_OUT,110414.71,26845.41,0.0,65,0
47,CASH_OUT,56953.90,1942.02,0.0,65,0
51,CASH_OUT,23261.30,20411.53,0.0,65,0
60,CASH_OUT,82940.31,3017.87,0.0,65,0
72,CASH_OUT,94253.33,25203.05,0.0,65,0


In [27]:
def get_reasons(row):
    reasons = []

    if row["risk_type"] == 1:
        reasons.append("High-risk transaction type")

    if row["high_amount"] == 1:
        reasons.append("High transaction amount")

    if row["balance_drained"] == 1:
        reasons.append("Sender balance drained after transaction")

    return reasons

df["reasons"] = df.apply(get_reasons, axis=1)

df[[
    "type",
    "amount",
    "risk_score",
    "decision",
    "reasons",
    "isFraud"
]].head(20)

,type,amount,risk_score,decision,reasons,isFraud
0,PAYMENT,9839.64,0,ALLOW,[],0
1,PAYMENT,1864.28,0,ALLOW,[],0
2,TRANSFER,181.00,65,REVIEW,"[High-risk transaction type, Sender balance dr...",1
3,CASH_OUT,181.00,65,REVIEW,"[High-risk transaction type, Sender balance dr...",1
4,PAYMENT,11668.14,0,ALLOW,[],0
5,PAYMENT,7817.71,0,ALLOW,[],0
6,PAYMENT,7107.77,0,ALLOW,[],0
7,PAYMENT,7861.64,0,ALLOW,[],0
8,PAYMENT,4024.36,35,ALLOW,[Sender balance drained after transaction],0
9,DEBIT,5337.77,0,ALLOW,[],0


In [28]:
sample_df = df.sample(n=50000, random_state=42)

sample_df.shape

(50000, 17)

In [ ]:
from app.rule_engine import evaluate_transaction

# Rule engine now lives in app/rule_engine.py so the notebook and API use the same logic.

In [ ]:
The rule-based scoring logic has been moved into app/rule_engine.py.

This notebook now serves as exploration and validation only, while the API imports the shared rule engine directly.